# Optimization Capstone: Fast GEMM

Bring together every optimization technique from the course to build a fast general matrix multiply (GEMM). Compare against cuBLAS and learn when to write custom kernels.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/cuda-optimization-capstone)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Then run the install cell below.

In [ ]:
!pip install numba

---

## Lesson examples (optional)

These are the code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.

### Lesson example: Putting It All Together

In [ ]:
!pip install numba
import numpy as np
from numba import cuda
import numba
import time
import math

# ============================================================
# Naive GEMM (baseline: no optimizations)
# ============================================================
@cuda.jit
def gemm_naive(A, B, C, M, N, K):
    """C = A @ B where A is MxK, B is KxN, C is MxN.
    One thread computes one element of C.
    No shared memory, no tiling.
    """
    row = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
    col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    
    if row < M and col < N:
        total = 0.0
        for k in range(K):
            total += A[row, k] * B[k, col]
        C[row, col] = total


# ============================================================
# Tiled GEMM (shared memory, Module 4 technique)
# ============================================================
TILE = 16

@cuda.jit
def gemm_tiled(A, B, C, M, N, K):
    """Tiled GEMM using shared memory.
    Each block computes a TILE x TILE sub-matrix of C.
    """
    sA = cuda.shared.array((TILE, TILE), dtype=numba.float32)
    sB = cuda.shared.array((TILE, TILE), dtype=numba.float32)
    
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    row = ty + cuda.blockIdx.y * TILE
    col = tx + cuda.blockIdx.x * TILE
    
    total = 0.0
    
    # Iterate over tiles along K dimension
    for t in range((K + TILE - 1) // TILE):
        # Load tile of A and B into shared memory
        a_col = t * TILE + tx
        b_row = t * TILE + ty
        
        if row < M and a_col < K:
            sA[ty, tx] = A[row, a_col]
        else:
            sA[ty, tx] = 0.0
        
        if b_row < K and col < N:
            sB[ty, tx] = B[b_row, col]
        else:
            sB[ty, tx] = 0.0
        
        cuda.syncthreads()
        
        # Compute partial dot product from this tile
        for k in range(TILE):
            total += sA[ty, k] * sB[k, tx]
        
        cuda.syncthreads()
    
    if row < M and col < N:
        C[row, col] = total


# ============================================================
# Benchmark: Naive vs Tiled
# ============================================================
def run_gemm_benchmark(M, N, K, iters=50):
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    
    d_A = cuda.to_device(A)
    d_B = cuda.to_device(B)
    d_C = cuda.device_array((M, N), dtype=np.float32)
    
    # Verify correctness
    C_ref = A @ B
    
    # Naive
    threads_naive = (16, 16)
    blocks_naive = ((N + 15) // 16, (M + 15) // 16)
    gemm_naive[blocks_naive, threads_naive](d_A, d_B, d_C, M, N, K)
    C_naive = d_C.copy_to_host()
    naive_ok = np.allclose(C_naive, C_ref, rtol=1e-3, atol=1e-3)
    
    # Tiled
    threads_tiled = (TILE, TILE)
    blocks_tiled = ((N + TILE - 1) // TILE, (M + TILE - 1) // TILE)
    gemm_tiled[blocks_tiled, threads_tiled](d_A, d_B, d_C, M, N, K)
    C_tiled = d_C.copy_to_host()
    tiled_ok = np.allclose(C_tiled, C_ref, rtol=1e-3, atol=1e-3)
    
    # Benchmark
    cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(iters):
        gemm_naive[blocks_naive, threads_naive](d_A, d_B, d_C, M, N, K)
    cuda.synchronize()
    naive_time = (time.perf_counter() - start) / iters
    
    start = time.perf_counter()
    for _ in range(iters):
        gemm_tiled[blocks_tiled, threads_tiled](d_A, d_B, d_C, M, N, K)
    cuda.synchronize()
    tiled_time = (time.perf_counter() - start) / iters
    
    flops = 2.0 * M * N * K
    naive_gflops = flops / naive_time / 1e9
    tiled_gflops = flops / tiled_time / 1e9
    
    return naive_time, tiled_time, naive_gflops, tiled_gflops, naive_ok, tiled_ok


print("=" * 70)
print("GEMM Optimization Review: Naive vs Tiled")
print("=" * 70)
print(f"{'Size':>12} {'Naive(ms)':>10} {'Tiled(ms)':>10} {'Naive GF/s':>11} "
      f"{'Tiled GF/s':>11} {'Speedup':>8} {'Correct':>8}")
print("-" * 72)

for size in [256, 512, 1024, 2048]:
    nt, tt, ng, tg, nok, tok = run_gemm_benchmark(size, size, size)
    print(f"{size:>5}x{size:<5} {nt*1e3:>10.2f} {tt*1e3:>10.2f} {ng:>11.1f} "
          f"{tg:>11.1f} {nt/tt:>7.1f}x {'OK' if nok and tok else 'FAIL':>7}")

print("\nTiling provides significant speedup by reusing data in shared memory.")
print("Next: register tiling and double buffering for even more performance.")

### Lesson example: Building a Fast GEMM

In [ ]:
!pip install numba
import numpy as np
from numba import cuda
import numba
import time
import math

# ============================================================
# Level 1: Naive GEMM (baseline)
# ============================================================
@cuda.jit
def gemm_naive(A, B, C, M, N, K):
    row = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
    col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if row < M and col < N:
        total = 0.0
        for k in range(K):
            total += A[row, k] * B[k, col]
        C[row, col] = total

# ============================================================
# Level 2: Tiled GEMM (shared memory)
# ============================================================
TILE = 16

@cuda.jit
def gemm_tiled(A, B, C, M, N, K):
    sA = cuda.shared.array((16, 16), dtype=numba.float32)
    sB = cuda.shared.array((16, 16), dtype=numba.float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    row = ty + cuda.blockIdx.y * 16
    col = tx + cuda.blockIdx.x * 16
    total = 0.0
    
    for t in range((K + 15) // 16):
        ac = t * 16 + tx
        br = t * 16 + ty
        sA[ty, tx] = A[row, ac] if (row < M and ac < K) else 0.0
        sB[ty, tx] = B[br, col] if (br < K and col < N) else 0.0
        cuda.syncthreads()
        for k in range(16):
            total += sA[ty, k] * sB[k, tx]
        cuda.syncthreads()
    
    if row < M and col < N:
        C[row, col] = total

# ============================================================
# Level 3: Register-tiled GEMM
# Each thread computes a 2x2 sub-tile of C
# Block tile: 32x32, thread tile: 2x2, so 16x16 threads per block
# ============================================================
BM = 32  # Block tile M
BN = 32  # Block tile N
BK = 16  # Block tile K
TM = 2   # Thread tile M
TN = 2   # Thread tile N

@cuda.jit
def gemm_register_tiled(A, B, C, M, N, K):
    """Register-tiled GEMM.
    Block: 32x32 output tile, using 16x16 threads.
    Each thread computes a 2x2 sub-tile (TM=2, TN=2).
    """
    sA = cuda.shared.array((32, 16), dtype=numba.float32)  # BM x BK
    sB = cuda.shared.array((16, 32), dtype=numba.float32)  # BK x BN
    
    tx = cuda.threadIdx.x  # 0..15
    ty = cuda.threadIdx.y  # 0..15
    
    # Global row/col for this thread's 2x2 output tile
    row_base = cuda.blockIdx.y * 32 + ty * 2
    col_base = cuda.blockIdx.x * 32 + tx * 2
    
    # Accumulators in registers (2x2)
    acc00 = 0.0
    acc01 = 0.0
    acc10 = 0.0
    acc11 = 0.0
    
    for t in range((K + 15) // 16):
        # Load A tile (32 x 16) with 16x16 = 256 threads
        # Each thread loads 2 elements (32 rows / 16 threads = 2)
        for i in range(2):
            a_row = cuda.blockIdx.y * 32 + ty * 2 + i
            a_col = t * 16 + tx
            if a_row < M and a_col < K:
                sA[ty * 2 + i, tx] = A[a_row, a_col]
            else:
                sA[ty * 2 + i, tx] = 0.0
        
        # Load B tile (16 x 32) with 16x16 = 256 threads
        # Each thread loads 2 elements (32 cols / 16 threads = 2)
        for j in range(2):
            b_row = t * 16 + ty
            b_col = cuda.blockIdx.x * 32 + tx * 2 + j
            if b_row < K and b_col < N:
                sB[ty, tx * 2 + j] = B[b_row, b_col]
            else:
                sB[ty, tx * 2 + j] = 0.0
        
        cuda.syncthreads()
        
        # Compute: each thread does 2x2 outer products
        for k in range(16):
            a0 = sA[ty * 2, k]
            a1 = sA[ty * 2 + 1, k]
            b0 = sB[k, tx * 2]
            b1 = sB[k, tx * 2 + 1]
            acc00 += a0 * b0
            acc01 += a0 * b1
            acc10 += a1 * b0
            acc11 += a1 * b1
        
        cuda.syncthreads()
    
    # Write results
    if row_base < M and col_base < N:
        C[row_base, col_base] = acc00
    if row_base < M and col_base + 1 < N:
        C[row_base, col_base + 1] = acc01
    if row_base + 1 < M and col_base < N:
        C[row_base + 1, col_base] = acc10
    if row_base + 1 < M and col_base + 1 < N:
        C[row_base + 1, col_base + 1] = acc11


# ============================================================
# Benchmark all levels
# ============================================================
def benchmark_gemm(M, N, K, iters=50):
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    C_ref = A @ B
    
    d_A = cuda.to_device(A)
    d_B = cuda.to_device(B)
    d_C = cuda.device_array((M, N), dtype=np.float32)
    flops = 2.0 * M * N * K
    
    results = []
    
    # Naive
    blocks = ((N + 15) // 16, (M + 15) // 16)
    gemm_naive[blocks, (16, 16)](d_A, d_B, d_C, M, N, K)
    ok = np.allclose(d_C.copy_to_host(), C_ref, rtol=1e-3, atol=1e-2)
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        gemm_naive[blocks, (16, 16)](d_A, d_B, d_C, M, N, K)
    cuda.synchronize()
    t = (time.perf_counter() - start) / iters
    results.append(("Naive", t, flops/t/1e9, ok))
    
    # Tiled
    blocks = ((N + 15) // 16, (M + 15) // 16)
    gemm_tiled[blocks, (16, 16)](d_A, d_B, d_C, M, N, K)
    ok = np.allclose(d_C.copy_to_host(), C_ref, rtol=1e-3, atol=1e-2)
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        gemm_tiled[blocks, (16, 16)](d_A, d_B, d_C, M, N, K)
    cuda.synchronize()
    t = (time.perf_counter() - start) / iters
    results.append(("Tiled", t, flops/t/1e9, ok))
    
    # Register-tiled
    blocks = ((N + 31) // 32, (M + 31) // 32)
    gemm_register_tiled[blocks, (16, 16)](d_A, d_B, d_C, M, N, K)
    ok = np.allclose(d_C.copy_to_host(), C_ref, rtol=1e-3, atol=1e-2)
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        gemm_register_tiled[blocks, (16, 16)](d_A, d_B, d_C, M, N, K)
    cuda.synchronize()
    t = (time.perf_counter() - start) / iters
    results.append(("Reg-Tiled", t, flops/t/1e9, ok))
    
    return results

print("=" * 70)
print("Progressive GEMM Optimization")
print("=" * 70)

for size in [512, 1024, 2048]:
    print(f"\nMatrix size: {size}x{size}")
    print(f"{'Kernel':>12} {'Time(ms)':>10} {'GFLOP/s':>10} {'% Peak':>8} {'OK':>4}")
    print("-" * 48)
    results = benchmark_gemm(size, size, size)
    peak = 8100  # T4 approximate peak fp32 GFLOP/s
    for name, t, gf, ok in results:
        print(f"{name:>12} {t*1e3:>10.2f} {gf:>10.1f} {gf/peak*100:>7.1f}% {'Y' if ok else 'N':>3}")

### Lesson example: Benchmarking and Profiling

In [ ]:
!pip install numba
import numpy as np
from numba import cuda
import numba
import time

# ============================================================
# All three GEMM levels
# ============================================================
@cuda.jit
def gemm_naive(A, B, C, M, N, K):
    row = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
    col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if row < M and col < N:
        total = 0.0
        for k in range(K):
            total += A[row, k] * B[k, col]
        C[row, col] = total

@cuda.jit
def gemm_tiled(A, B, C, M, N, K):
    sA = cuda.shared.array((16, 16), dtype=numba.float32)
    sB = cuda.shared.array((16, 16), dtype=numba.float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    row = ty + cuda.blockIdx.y * 16
    col = tx + cuda.blockIdx.x * 16
    total = 0.0
    for t in range((K + 15) // 16):
        ac = t * 16 + tx
        br = t * 16 + ty
        sA[ty, tx] = A[row, ac] if (row < M and ac < K) else 0.0
        sB[ty, tx] = B[br, col] if (br < K and col < N) else 0.0
        cuda.syncthreads()
        for k in range(16):
            total += sA[ty, k] * sB[k, tx]
        cuda.syncthreads()
    if row < M and col < N:
        C[row, col] = total

@cuda.jit
def gemm_reg_tiled(A, B, C, M, N, K):
    sA = cuda.shared.array((32, 16), dtype=numba.float32)
    sB = cuda.shared.array((16, 32), dtype=numba.float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    row_base = cuda.blockIdx.y * 32 + ty * 2
    col_base = cuda.blockIdx.x * 32 + tx * 2
    acc00 = 0.0; acc01 = 0.0; acc10 = 0.0; acc11 = 0.0
    for t in range((K + 15) // 16):
        for i in range(2):
            ar = cuda.blockIdx.y * 32 + ty * 2 + i
            ac = t * 16 + tx
            sA[ty*2+i, tx] = A[ar, ac] if (ar < M and ac < K) else 0.0
        for j in range(2):
            br = t * 16 + ty
            bc = cuda.blockIdx.x * 32 + tx * 2 + j
            sB[ty, tx*2+j] = B[br, bc] if (br < K and bc < N) else 0.0
        cuda.syncthreads()
        for k in range(16):
            a0 = sA[ty*2, k]; a1 = sA[ty*2+1, k]
            b0 = sB[k, tx*2]; b1 = sB[k, tx*2+1]
            acc00 += a0*b0; acc01 += a0*b1; acc10 += a1*b0; acc11 += a1*b1
        cuda.syncthreads()
    if row_base < M and col_base < N:
        C[row_base, col_base] = acc00
    if row_base < M and col_base+1 < N:
        C[row_base, col_base+1] = acc01
    if row_base+1 < M and col_base < N:
        C[row_base+1, col_base] = acc10
    if row_base+1 < M and col_base+1 < N:
        C[row_base+1, col_base+1] = acc11

# ============================================================
# Comprehensive benchmark
# ============================================================
def benchmark_all(M, N, K, iters=50):
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    C_ref = A @ B
    d_A, d_B = cuda.to_device(A), cuda.to_device(B)
    d_C = cuda.device_array((M, N), dtype=np.float32)
    flops = 2.0 * M * N * K
    
    configs = [
        ("Naive",     gemm_naive,     (16,16), ((N+15)//16, (M+15)//16)),
        ("Tiled-16",  gemm_tiled,     (16,16), ((N+15)//16, (M+15)//16)),
        ("RegTile-2x2", gemm_reg_tiled, (16,16), ((N+31)//32, (M+31)//32)),
    ]
    
    results = []
    for name, kernel, threads, blocks in configs:
        # Correctness
        kernel[blocks, threads](d_A, d_B, d_C, M, N, K)
        ok = np.allclose(d_C.copy_to_host(), C_ref, rtol=1e-3, atol=1e-2)
        
        # Timing
        cuda.synchronize()
        start = time.perf_counter()
        for _ in range(iters):
            kernel[blocks, threads](d_A, d_B, d_C, M, N, K)
        cuda.synchronize()
        t = (time.perf_counter() - start) / iters
        gf = flops / t / 1e9
        results.append((name, t, gf, ok))
    
    # NumPy (CPU BLAS) for reference
    start = time.perf_counter()
    for _ in range(min(iters, 20)):
        _ = A @ B
    cpu_time = (time.perf_counter() - start) / min(iters, 20)
    cpu_gf = flops / cpu_time / 1e9
    results.append(("NumPy(CPU)", cpu_time, cpu_gf, True))
    
    # CuPy (cuBLAS) comparison
    try:
        import cupy as cp
        A_cp = cp.asarray(A)
        B_cp = cp.asarray(B)
        # Warm up cuBLAS
        _ = cp.matmul(A_cp, B_cp)
        cp.cuda.Device(0).synchronize()
        start = time.perf_counter()
        for _ in range(iters):
            C_cp = cp.matmul(A_cp, B_cp)
        cp.cuda.Device(0).synchronize()
        cublas_time = (time.perf_counter() - start) / iters
        cublas_gf = flops / cublas_time / 1e9
        results.append(("cuBLAS(CuPy)", cublas_time, cublas_gf, True))
    except ImportError:
        print("  (Install CuPy for cuBLAS comparison: pip install cupy-cuda12x)")
    
    return results

print("=" * 75)
print("GEMM Performance Progression & Comparison")
print("=" * 75)
print("T4 peak FP32: ~8,100 GFLOP/s")

try:
    import cupy
    print("CuPy detected -- cuBLAS comparison enabled!")
except ImportError:
    print("Install CuPy for cuBLAS comparison: pip install cupy-cuda12x")

for size in [256, 512, 1024, 2048]:
    print(f"\n--- {size}x{size} GEMM ({2*size**3/1e9:.2f} GFLOP) ---")
    print(f"{'Kernel':>14} {'Time(ms)':>10} {'GFLOP/s':>10} {'%Peak':>7} {'OK':>4}")
    print("-" * 48)
    results = benchmark_all(size, size, size)
    peak = 8100.0
    for name, t, gf, ok in results:
        pct = f"{gf/peak*100:.1f}%" if 'CPU' not in name else 'N/A'
        print(f"{name:>14} {t*1e3:>10.2f} {gf:>10.1f} {pct:>7} {'Y' if ok else 'N':>3}")

print("\n" + "=" * 75)
print("Key Takeaways")
print("=" * 75)
print("1. Tiling gives 5-10x over naive (shared memory reuse)")
print("2. Register tiling gives another 1.5-3x (compute-to-load ratio)")
print("3. cuBLAS (via CuPy/PyTorch) reaches 80-90% of peak")
print("4. Our best Numba kernel reaches ~15-30% of peak")
print("5. The gap shows the value of low-level optimizations")
print("   (warp-level ops, bank conflict avoidance, instruction scheduling)")
print("6. For production GEMM: ALWAYS use cuBLAS")
print("7. Custom kernels shine for FUSED operations (GEMM + bias + ReLU)")

---

## Exercise: Progressive GEMM Optimization and Fused Kernel


In [ ]:
import numpy as np
from numba import cuda
import numba
import time

# ============================================================
# PART A: GEMM Progression
# ============================================================

# TODO 1: Naive GEMM
@cuda.jit
def gemm_naive(A, B, C, M, N, K):
    """C = A @ B. One thread per output element. No shared memory."""
    row = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
    col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    # TODO: Compute C[row, col] = sum(A[row, k] * B[k, col] for k in range(K))
    pass


# TODO 2: Tiled GEMM (16x16 tiles)
@cuda.jit
def gemm_tiled(A, B, C, M, N, K):
    """Tiled GEMM with 16x16 shared memory tiles."""
    sA = cuda.shared.array((16, 16), dtype=numba.float32)
    sB = cuda.shared.array((16, 16), dtype=numba.float32)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    row = ty + cuda.blockIdx.y * 16
    col = tx + cuda.blockIdx.x * 16
    total = 0.0
    # TODO: Iterate over tiles along K dimension
    # For each tile:
    #   1. Load tile of A and B into shared memory (with bounds checking)
    #   2. syncthreads()
    #   3. Compute partial dot product from this tile
    #   4. syncthreads()
    pass


# TODO 3: Register-tiled GEMM
@cuda.jit
def gemm_reg_tiled(A, B, C, M, N, K):
    """Register-tiled GEMM. 16x16 threads compute 32x32 output (2x2 per thread)."""
    sA = cuda.shared.array((32, 16), dtype=numba.float32)
    sB = cuda.shared.array((16, 32), dtype=numba.float32)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    # TODO: 4 accumulators, tile loop, load A(32x16)/B(16x32), compute 4 FMAs per k
    pass


# ============================================================
# PART B: Fused GEMM + Bias + ReLU
# ============================================================

# TODO 4: Separate operations (baseline for comparison)
@cuda.jit
def bias_relu_kernel(C, bias, output, M, N):
    """Separate kernel: output = max(0, C + bias)."""
    row = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
    col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    # TODO: output[row, col] = max(0.0, C[row, col] + bias[col])
    pass


# TODO 5: Fused GEMM + bias + ReLU
@cuda.jit
def gemm_bias_relu_fused(A, B, bias, output, M, N, K):
    """Fused: output = max(0, A @ B + bias) in ONE kernel.
    Uses tiled GEMM (16x16) with bias + ReLU applied before writing.
    """
    sA = cuda.shared.array((16, 16), dtype=numba.float32)
    sB = cuda.shared.array((16, 16), dtype=numba.float32)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    row = ty + cuda.blockIdx.y * 16
    col = tx + cuda.blockIdx.x * 16
    total = 0.0
    # TODO: Same tiled GEMM as above, but after computing total:
    #   total += bias[col]           # Add bias
    #   output[row, col] = max(0.0, total)  # ReLU + write
    # The key: intermediate GEMM result stays in registers!
    pass


# ============================================================
# Benchmarking Framework
# ============================================================
def benchmark_kernel(kernel, blocks, threads, args, name, ref, iters=50):
    """Benchmark any kernel and check correctness."""
    kernel[blocks, threads](*args)
    cuda.synchronize()
    ok = np.allclose(args[2].copy_to_host() if len(args) == 6 
                     else args[3].copy_to_host(), ref, rtol=1e-3, atol=1e-2)
    start = time.perf_counter()
    for _ in range(iters):
        kernel[blocks, threads](*args)
    cuda.synchronize()
    avg_time = (time.perf_counter() - start) / iters
    return avg_time, ok


# ============================================================
# Part A: GEMM Benchmark
# ============================================================
print("PART A: GEMM Performance Progression")
print("=" * 55)
for size in [512, 1024]:
    M = N = K = size
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    C_ref = A @ B
    d_A, d_B = cuda.to_device(A), cuda.to_device(B)
    d_C = cuda.device_array((M, N), dtype=np.float32)
    flops = 2.0 * M * N * K
    
    print(f"\n{size}x{size} GEMM:")
    # TODO: Benchmark naive, tiled, register-tiled
    # Print: name, time(ms), GFLOP/s, %peak


# ============================================================
# Part B: Fused vs Separate
# ============================================================
print("\n\nPART B: Fusion Benefit")
print("=" * 55)
M, N, K = 1024, 1024, 1024
A = np.random.randn(M, K).astype(np.float32)
B = np.random.randn(K, N).astype(np.float32)
bias = np.random.randn(N).astype(np.float32)
ref = np.maximum(0, A @ B + bias)

d_A, d_B = cuda.to_device(A), cuda.to_device(B)
d_bias = cuda.to_device(bias)

# TODO: Time separate approach (gemm_tiled + bias_relu_kernel)
# TODO: Time fused approach (gemm_bias_relu_fused)
# TODO: Compare times and compute memory traffic saved
# Memory saved = 2 * M * N * 4 bytes (two eliminated intermediates)

## Your tasks

Build progressively optimized GEMM kernels, benchmark them, and then implement a **fused GEMM + bias + ReLU kernel** -- the practical payoff of everything you have learned.

**Part A: GEMM Progression (benchmarking)**
1. Implement a naive GEMM kernel (one thread per output element, no shared memory)
2. Implement a tiled GEMM kernel (16x16 shared memory tiles)
3. Implement a register-tiled GEMM kernel (each thread computes a 2x2 sub-tile)
4. Verify correctness and measure GFLOP/s at sizes 512, 1024, 2048

**Part B: Fused GEMM + Bias + ReLU (the real exercise)**
5. Implement a fused kernel that computes `output = max(0, A @ B + bias)` in a SINGLE kernel
6. Compare the fused kernel against separate operations (GEMM then add bias then ReLU)
7. Measure and report: time saved by fusion, memory traffic saved

**Why fusion matters**: In a neural network forward pass, GEMM is followed by bias addition and activation. Without fusion, the GEMM output (M*N floats) is written to global memory, read back for bias add, written again, read back for ReLU, and written a third time. With fusion, the intermediate results stay in registers -- saving 2 round-trips to global memory.

**Hints:**
- FLOPs for MxN @ NxK = 2 * M * N * K
- For the fused kernel: compute the dot product in the inner loop, add bias[col] after, apply max(0.0, val) before writing to output
- Memory traffic saved by fusion: 2 * M * N * 4 bytes (two eliminated intermediate writes/reads)
- Always warm up kernels before timing (first call includes JIT compilation)
- Use cuda.synchronize() before starting and stopping the timer
- T4 peak: approximately 8,100 GFLOP/s FP32

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/cuda-optimization-capstone) for the solution and discussion.